# Getting started
This notebook can be used for local testing and development of the biodata registry prototype.

## Running the services with Docker
From the biodata-registry-sandbox folder, run
```
docker compose up --build -d
```
To build the containers and run them in the background. It may take several seconds for the services to spin up. You can check the status with `docker ps`. You should see something similar to:
```
CONTAINER ID   IMAGE                             COMMAND                  CREATED          STATUS          PORTS                                                             NAMES
68724fbdbc25   biodata-registry-sandbox-lambda   "python -u sync_regi…"   20 seconds ago   Up 3 seconds                                                                      lambda
b9aa0392f294   quay.io/debezium/connect:3.1      "/docker-entrypoint.…"   20 seconds ago   Up 20 seconds   8778/tcp, 0.0.0.0:8083->8083/tcp, [::]:8083->8083/tcp, 9092/tcp   biodata-registry-sandbox-connect-1
ffebe7e06b0e   quay.io/debezium/kafka:3.1        "/docker-entrypoint.…"   20 seconds ago   Up 20 seconds   9092/tcp, 0.0.0.0:29092->29092/tcp, [::]:29092->29092/tcp         biodata-registry-sandbox-kafka-1
eb72a40e0a86   biodata-registry-sandbox-api      "fastapi run src/bio…"   20 seconds ago   Up 20 seconds   0.0.0.0:5000->80/tcp, [::]:5000->80/tcp                           registry_api
ac65b340cefe   mongo:latest                      "docker-entrypoint.s…"   20 seconds ago   Up 20 seconds   0.0.0.0:27017->27017/tcp, [::]:27017->27017/tcp                   mongodb
c4b78e903648   quay.io/debezium/zookeeper:3.1    "/docker-entrypoint.…"   20 seconds ago   Up 20 seconds   2888/tcp, 0.0.0.0:2181->2181/tcp, [::]:2181->2181/tcp, 3888/tcp   biodata-registry-sandbox-zookeeper-1
090b9c049098   debezium/postgres:15              "docker-entrypoint.s…"   20 seconds ago   Up 20 seconds   0.0.0.0:5432->5432/tcp, [::]:5432->5432/tcp                       biodata-registry-sandbox-postgres-1
```
In particular, the `lambda` service is the last thing that spins up.

## Install the client
There is an autogenerated python client that is used to interface with the API server. Create a python environment with venv, conda, or uv and then navigate into the `client` folder and pip install it:
```
pip install -e .
```

## Populate Databases
The following script will populate the database by migrating 400 records from DocDB.

In [ ]:
# TODO: Add tqdm progress bar
# This will take a minute to run.

In [ ]:
%run populate_database.py

# Using the Python client

The python client can be used to get, add, update, and delete records from the database.

In [ ]:
# Configuring the client
from biodata_registry_api_client import (
    Configuration,
    ApiClient,
    CoreApi,
    ViewsApi,
)

configuration = Configuration(
    host = "http://localhost:5000"
)

api_client = ApiClient(configuration)
admin_api = AdminApi(api_client)
core_api = CoreApi(api_client)
views_api = ViewsApi(api_client)


In [ ]:
# Fetching a single row
user = admin_api.get_user(id=1)
subject = core_api.get_subject(id=1)
data_asset_view = views_api.get_data_asset_view(data_asset_id=1)

print(user)
print(subject)
print(data_asset_view)

In [ ]:
# Fetching multiple rows
subjects = core_api.get_subjects(limit=100, offset=0)
print(len(subjects))
print(subjects[0])
print(subjects[-1])


In [ ]:
# Fetching a schema
# TODO: add name to get_schemas API
schemas = core_api.get_schemas()
subject_schema = [schema for schema in schemas if schema.name == "subject"][0]
print(subject_schema)

In [ ]:
# Creating a record
from biodata_registry_api_client import SubjectCreate

subject = core_api.get_subject(id=1)
subject_data = {
  'notes': " (SubjectUpgraderV1V2): Source not specified, defaulting to 'Other'.", 
  'subject_id': 'abc123', 
  'describedBy': 'https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/subject.py', 
  'object_type': 'Subject', 
  'schema_version': '2.2.1', 
  'subject_details': {
    'sex': 'Male', 
    'rrid': None, 
    'source': {
      'name': 'Other', 
      'registry': None, 
      'abbreviation': None, 
      'registry_identifier': None
    }, 
    'strain': {
      'name': 'C57BL/6J', 
      'species': 'Mus musculus', 
      'registry': 'Mouse Genome Informatics (MGI)', 
      'registry_identifier': 'MGI:3028467'
    }, 
    'alleles': [], 
    'housing': None, 
    'species': {
      'name': 'Mus musculus', 
      'registry': 'National Center for Biotechnology Information (NCBI)', 
      'common_name': 'House mouse', 
      'registry_identifier': 'NCBI:txid10090'
    }, 
    'genotype': 'wt/wt ', 
    'object_type': 'Mouse subject', 
    'restrictions': None, 
    'breeding_info': {
      'maternal_id': 'None', 
      'object_type': 'Breeding info', 
      'paternal_id': 'None', 
      'breeding_group': None, 
      'maternal_genotype': 'None', 
      'paternal_genotype': 'None'
    }, 
    'date_of_birth': '2023-10-03', 
    'wellness_reports': []
  }
}
subject_create = SubjectCreate(
    name = "abc123",
    schema_id=subject_schema.id,
    space_id=1,
    data=subject_data
) 
new_subject_record = core_api.create_subject(subject_create=subject_create)

In [ ]:
print(new_subject_record)

In [ ]:
# Updating a record
from biodata_registry_api_client import SubjectUpdate

subject_update = SubjectUpdate(**new_subject_record.model_dump(mode="json"))
record_id = new_subject_record.id
subject_update.data["notes"] = "Hello World!"
updated_subject_record = core_api.update_subject(id=record_id, subject_update=subject_update)
print(updated_subject_record)


In [ ]:
# Deleting a record
record_id_to_delete = updated_subject_record.id
delete_response = core_api.delete_subject(id=record_id_to_delete)
print(delete_response)

In [ ]:
# Adding a data asset to a collection

my_collection = admin_api.get_collection(id=1)
my_data_asset = core_api.get_data_asset(id=2)
my_collection_data_assets = admin_api.get_collection_data_assets(id=1)
my_data_asset_collections = core_api.get_data_asset_collections(id=2)
print(my_collection)
print(my_data_asset)
print(my_collection_data_assets)
print(my_data_asset_collections)

# Two methods
add_data_asset_response = admin_api.put_collection_data_asset(id=1, data_asset_id=2)
# add_data_asset_response = core_api.put_data_asset_collection(id=2, collection_id=1)
print(add_data_asset_response)

In [ ]:
my_collection_data_assets = admin_api.get_collection_data_assets(id=1)
my_data_asset_collections = core_api.get_data_asset_collections(id=2)
print(my_collection_data_assets)
print(my_data_asset_collections)

In [ ]:
# Removing a data asset from a collection (Two methods)
rm_data_asset_response = admin_api.remove_collection_data_asset(id=1, data_asset_id=2)
print(rm_data_asset_response)


In [ ]:
my_collection_data_assets = admin_api.get_collection_data_assets(id=1)
my_data_asset_collections = core_api.get_data_asset_collections(id=2)
print(my_collection_data_assets)
print(my_data_asset_collections)